# 🌟 萬象 Wan 2.1-I2V-14B 滿血 5 秒換裝動態工作流 (Google Colab A100 / L4 專用)

歡迎使用 **Wan 2.1 140 億參數量極致畫質視頻生成工作流**！

### ⚡ 運行環境建議
1. 請在頂部選單點選 **「執行階段 (Runtime)」** ➔ **「變更執行階段類型 (Change runtime type)」**。
2. **硬體加速器**：選擇 **A100 GPU**（或 L4 GPU）。
3. **執行階段資源**：開啟 **大量 RAM (High-RAM)** 開關。
4. 點選 **「全部執行 (Run All)」**(快速鍵 `Ctrl + F9`)。

### 🚀 預期成效
- **時長與規格**：滿血 81 幀 / 16 FPS = **5.06 秒標準動態短影音**。
- **人物品質**：以 98 分評級首選母圖 `portrait_2.jpg` 定錨，三位法越混血仙女絕美絕倫，真絲晨袍自然滑落、蕾絲肩帶細膩微調、自然胸腔呼吸，徹底杜絕肢體畸變與溶面。
- **極速生成**：A100 大顯存全精度加持下，約 **2 ~ 4 分鐘** 即可出片！
- **自動下載**：生成完成後會自動打包並觸發瀏覽器下載至本地 `Downloads/`，本地 watcher 會自動同步至 `outputs/` 並更新畫廊！

In [ ]:
# =========================================================================
# 1. 硬體加速與記憶體檢測 (A100 / L4 GPU + High-RAM 檢查)
# =========================================================================
import torch, psutil

if not torch.cuda.is_available():
    raise RuntimeError('❌ 未檢測到 GPU！請在 Colab 頂部「執行階段」->「變更執行階段類型」選擇 A100 或 L4 GPU。')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)

print('✅ 雲端硬體檢測通過！')
print(f'   - 顯示卡: {gpu_name} ({vram_gb} GB VRAM)')
print(f'   - 系統 RAM: {ram_gb} GB')

if vram_gb >= 38.0:
    print('🔥 [滿血 A100 模式] 具備充足大顯存，將以純 GPU 常駐模式極速推理 (預計 2~3 分鐘出片)！')
else:
    print(f'⚡ [省顯存模式] 顯存 {vram_gb} GB，將自動啟動 CPU Offload 節省模式 (相容 L4 24GB)。')

In [ ]:
# =========================================================================
# 2. 安裝最新 Wan 2.1 影像生成依賴庫
# =========================================================================
!pip install -q "diffusers>=0.34.0" "transformers>=4.48.0" "accelerate" "imageio[ffmpeg]" "ftfy" opencv-python Pillow

In [ ]:
# =========================================================================
# 3. 載入並無損縮放母圖 (portrait_2.jpg - 98分評級)
# =========================================================================
import urllib.request, os
from PIL import Image, ImageOps

mother_url = 'https://raw.githubusercontent.com/Chenponpon/colab-ai-media-lab/main/assets/mother_images/portrait_2.jpg'
local_mother = 'input_mother.jpg'

print(f'📥 下載鎖定之 98 分母圖: {mother_url} ...')
urllib.request.urlretrieve(mother_url, local_mother)

orig_img = Image.open(local_mother).convert('RGB')
print(f'✅ 母圖載入成功，原始尺寸: {orig_img.size[0]}x{orig_img.size[1]}')

# 無損裁切居中至 Wan 2.1 標準 480x832 豎版規格 (不壓縮拉伸人物臉部比例)
target_w, target_h = 480, 832
input_img = ImageOps.fit(orig_img, (target_w, target_h), Image.Resampling.LANCZOS)
input_img.save('input_resized_480x832.jpg', quality=98)
print(f'✅ 已等比居中縮放至 Wan2.1 標準規格: {target_w}x{target_h}')

In [ ]:
# =========================================================================
# 4. 啟動萬象 Wan2.1-I2V-14B 全精度 5 秒換裝動態推理
# =========================================================================
import time, json, cv2
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video

model_id = 'Wan-AI/Wan2.1-I2V-14B-480P-Diffusers'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'⏳ 正在載入萬象 Wan 2.1 14B 視頻管線 ({model_id}, {dtype})...')

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
pipe = WanImageToVideoPipeline.from_pretrained(model_id, torch_dtype=dtype)

if vram_gb >= 38.0:
    print('🔥 啟用純 GPU 顯存駐留 (Zero PCIe Bottleneck)...')
    pipe.to('cuda')
else:
    print(f'⚡ 顯存為 {round(vram_gb, 1)} GB，啟用 Model CPU Offload 節省模式...')
    pipe.enable_model_cpu_offload()

# 啟用 VAE 分塊分片防爆顯存技術
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

# 閨房柔光換裝動態提示詞 (100% 合規不露點)
motion_prompt = (
    'Three breathtakingly gorgeous 20-year-old French-Vietnamese mixed-race women in an elegant Paris boudoir on a crisp white bed, '
    'delicate sheer silk morning robes gently slipping off smooth pale shoulders, revealing intricate sheer floral lace bodysuits artfully covering bust apex, '
    'hands naturally adjusting thin lingerie shoulder straps, gentle chest breathing movement, lively boudoir interaction with shared sweet smiles, '
    'cinematic warm morning lighting, photorealistic documentary quality, 8k, authentic physical motion of slipping silk fabric, perfectly stable angelic faces.'
)
negative_prompt = (
    'ugly, deformed, mutated limbs, missing fingers, extra hands, melting face, plastic skin, '
    'blurry, distorted eyes, jerky motion, bad anatomy, low quality, nude, nipples, exposed areola, uncensored'
)

total_frames = 81 # 81 幀 / 16 FPS = 5.06 秒滿血時長
target_fps = 16
num_steps = 25
guidance = 5.0
print(f'🎬 開始生成 Wan 2.1-I2V 視頻 ({total_frames} 幀 / {target_fps} FPS = 5.06 秒)...\n')

t0 = time.time()
generator = torch.Generator(device='cuda').manual_seed(42)

video_frames = pipe(
    image=input_img,
    prompt=motion_prompt,
    negative_prompt=negative_prompt,
    width=target_w,
    height=target_h,
    num_frames=total_frames,
    num_inference_steps=num_steps,
    guidance_scale=guidance,
    generator=generator
).frames[0]

dt = round(time.time() - t0, 2)
out_vid_name = 'changing_clothes_wan21_a100_5s.mp4'
export_to_video(video_frames, out_vid_name, fps=target_fps)
vid_sz_mb = round(os.path.getsize(out_vid_name) / (1024 * 1024), 2)
print(f'\n🎉 5.06 秒換裝視頻生成完成！總耗時: {dt} 秒，檔案大小: {vid_sz_mb} MB')

# 提取客觀驗收抽檢幀 (第 0 幀、中間幀、末尾幀)
cap = cv2.VideoCapture(out_vid_name)
real_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
real_fps = cap.get(cv2.CAP_PROP_FPS)
real_dur = round(real_frames / real_fps, 2) if real_fps > 0 else 0

ret, f_start = cap.read()
if ret: cv2.imwrite('real_frame_start.jpg', f_start)
cap.set(cv2.CAP_PROP_POS_FRAMES, real_frames // 2)
ret, f_mid = cap.read()
if ret: cv2.imwrite('real_frame_mid.jpg', f_mid)
cap.set(cv2.CAP_PROP_POS_FRAMES, real_frames - 1)
ret, f_end = cap.read()
if ret: cv2.imwrite('real_frame_end.jpg', f_end)
cap.release()

bench = {
    'engine': 'Wan-AI/Wan2.1-I2V-14B-480P (Google Colab A100 Full Precision)',
    'gpu': gpu_name,
    'elapsed_sec': dt,
    'frames': real_frames,
    'fps': real_fps,
    'duration_sec': real_dur,
    'resolution': f'{target_w}x{target_h}',
    'video_file': out_vid_name,
    'file_size_mb': vid_sz_mb
}
with open('video_benchmark_report.json', 'w', encoding='utf-8') as bf:
    json.dump(bench, bf, indent=2, ensure_ascii=False)
print(f'✅ 基準報告已產製: video_benchmark_report.json (實測時長: {real_dur} 秒 / {real_frames} 幀)')

In [ ]:
# =========================================================================
# 5. 打包成果並自動觸發瀏覽器下載至本地電腦
# =========================================================================
from google.colab import files
import zipfile, time

batch_ts = time.strftime('%Y%m%d_%H%M%S')
batch_zip = f'colab_wan21_a100_{batch_ts}.zip'

with zipfile.ZipFile(batch_zip, 'w') as zf:
    zf.write(out_vid_name)
    zf.write('real_frame_start.jpg')
    zf.write('real_frame_mid.jpg')
    zf.write('real_frame_end.jpg')
    zf.write('video_benchmark_report.json')
    if os.path.exists('input_mother.jpg'):
        zf.write('input_mother.jpg')

print(f'⬇️ 正在打包並觸發瀏覽器下載: {batch_zip} ...')
files.download(batch_zip)
print('\n🎉 下載指令已送出！本地的 auto-sync watcher 將在 2 秒內自動攔截、解壓至 outputs/ 並刷新畫廊展示！')